In [0]:
from pyspark.sql.functions import *

In [0]:
spark.conf.set(
    "fs.azure.account.key.stroadaccident01.dfs.core.windows.net",
    "<STORAGE_ACCOUNT_KEY>"
)

In [0]:
silver_path = "abfss://silver@stroadaccident01.dfs.core.windows.net/"
gold_path = "abfss://gold@stroadaccident01.dfs.core.windows.net/"

In [0]:
accident = spark.read.format("delta").load(f"{silver_path}accident_details")

location = spark.read.format("delta").load(f"{silver_path}location_details")

weather = spark.read.format("delta").load(f"{silver_path}weather_details")

road = spark.read.format("delta").load(f"{silver_path}road_features")

daylight = spark.read.format("delta").load(f"{silver_path}daylight_conditions")

In [0]:
state_summary = (
    accident.join(location, "ID")
    .groupBy("State")
    .agg(
        count("*").alias("Total_Accidents"),
        avg("Severity").alias("Average_Severity")
    )
)

state_summary.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.save(f"{gold_path}state_summary")

display(state_summary)

In [0]:
weather_summary = (
    accident.join(weather, "ID")
    .groupBy("Weather_Condition")
    .agg(
        count("*").alias("Total_Accidents")
    )
)

weather_summary.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.save(f"{gold_path}weather_summary")

display(weather_summary)

In [0]:
monthly_summary = (
    accident
    .withColumn("Year", year("Start_Time"))
    .withColumn("Month", month("Start_Time"))
    .groupBy("Year","Month")
    .agg(
        count("*").alias("Total_Accidents")
    )
    .orderBy("Year","Month")
)

monthly_summary.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.save(f"{gold_path}monthly_summary")

display(monthly_summary)

In [0]:
road_summary = (
    road.groupBy(
        "Traffic_Signal",
        "Junction",
        "Crossing",
        "Railway"
    )
    .agg(
        count("*").alias("Total_Accidents")
    )
)

road_summary.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.save(f"{gold_path}road_feature_summary")

display(road_summary)

In [0]:
daylight_summary = (
    accident.join(daylight,"ID")
    .groupBy("Sunrise_Sunset")
    .agg(
        count("*").alias("Total_Accidents")
    )
)

daylight_summary.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.save(f"{gold_path}daylight_summary")

display(daylight_summary)

In [0]:
gold_tables = [
    "state_summary",
    "weather_summary",
    "monthly_summary",
    "road_feature_summary",
    "daylight_summary"
]

for table in gold_tables:

    df = spark.read.format("delta").load(f"{gold_path}{table}")

    print("="*60)
    print(table)
    print(df.count())

    display(df)

In [0]:
state_summary.write.mode("overwrite").parquet("abfss://gold@stroadaccident01.dfs.core.windows.net/pbi/state_summary")

weather_summary.write.mode("overwrite").parquet("abfss://gold@stroadaccident01.dfs.core.windows.net/pbi/weather_summary")

monthly_summary.write.mode("overwrite").parquet("abfss://gold@stroadaccident01.dfs.core.windows.net/pbi/monthly_summary")

road_summary.write.mode("overwrite").parquet("abfss://gold@stroadaccident01.dfs.core.windows.net/pbi/road_feature_summary")

daylight_summary.write.mode("overwrite").parquet("abfss://gold@stroadaccident01.dfs.core.windows.net/pbi/daylight_summary")